In [20]:
from prophet import Prophet
import pandas as pd
import os

# ✅ Tạo thư mục nếu chưa có
os.makedirs("/mnt/data", exist_ok=True)

# ✅ Load lại dữ liệu gốc
df = pd.read_parquet("/mnt/data/merged_real_estate_macro.parquet")

# ✅ Chuẩn hóa cột ngày
df['ds'] = pd.to_datetime(df[['Year', 'Month']].assign(DAY=1))

# ✅ Lọc để lấy dữ liệu đến 2022-12-01 (dự báo từ 2023-01 trở đi)
df = df[df['ds'] <= "2022-12-01"]

# ✅ Trung bình giá mỗi tháng
monthly_df = df.groupby('ds')['Price'].mean().reset_index()
monthly_df.columns = ['ds', 'y']

# ✅ Train Prophet
model = Prophet()
model.fit(monthly_df)

# ✅ Dự báo từ 2023-01 đến 2023-06
future = model.make_future_dataframe(periods=6, freq='MS')  # MS = Month Start
forecast = model.predict(future)

# ✅ Lưu 6 tháng dự báo
forecast_prophet = forecast[['ds', 'yhat']].tail(6)
forecast_prophet.to_csv("/mnt/data/predict_prophet_v3.csv", index=False)

forecast_prophet.tail(10)


INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
DEBUG:cmdstanpy:input tempfile: /tmp/tmpngk5gnff/a1vc4s6k.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpngk5gnff/2rvhx6wm.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.11/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=29939', 'data', 'file=/tmp/tmpngk5gnff/a1vc4s6k.json', 'init=/tmp/tmpngk5gnff/2rvhx6wm.json', 'output', 'file=/tmp/tmpngk5gnff/prophet_modeltovtah4r/prophet_model-20250417100930.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
10:09:30 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
10:09:31 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


,ds,yhat
336,2023-01-01,404848.516175
337,2023-02-01,398308.367070
338,2023-03-01,408222.643583
339,2023-04-01,402610.574678
340,2023-05-01,400042.554345
341,2023-06-01,406898.014577


In [22]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense
import os

# ✅ Load lại data gốc từ parquet
df = pd.read_parquet("/mnt/data/merged_real_estate_macro.parquet")
df['ds'] = pd.to_datetime(df[['Year', 'Month']].assign(DAY=1))

# ✅ Lấy giá trung bình theo tháng
monthly_df = df.groupby('ds')['Price'].mean().reset_index()

# ✅ Chia train/test: dùng đến 2022-12 để train, 2023-01 → 2023-06 để predict
train = monthly_df[monthly_df['ds'] <= "2022-12-01"]
test_months = pd.date_range(start="2023-01-01", periods=6, freq='MS')

# ✅ Scale giá
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train[['Price']])

# ✅ Tạo chuỗi LSTM (dùng 12 tháng làm input)
def create_sequences(data, window=12):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled)

# ✅ Reshape lại thành [samples, timesteps, features]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))

# ✅ Build model đơn giản
model = Sequential()
model.add(LSTM(64, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
model.fit(X_train, y_train, epochs=200, verbose=0)

# ✅ Dự báo 6 tháng tiếp theo
last_sequence = train_scaled[-12:].reshape((1, 12, 1))
pred_scaled = []

for _ in range(6):
    pred = model.predict(last_sequence, verbose=0)
    pred_scaled.append(pred[0, 0])
    # ✅ FIX reshape đúng shape (1, 1, 1)
    last_sequence = np.concatenate([last_sequence[:, 1:, :], pred.reshape(1, 1, 1)], axis=1)


# ✅ Inverse transform về giá thật
pred_prices = scaler.inverse_transform(np.array(pred_scaled).reshape(-1, 1)).flatten()

# ✅ Save
forecast_dates = test_months
lstm_forecast_df = pd.DataFrame({'ds': forecast_dates, 'lstm_pred': pred_prices})
lstm_forecast_df.to_csv("/mnt/data/predict_lstm_v2.csv", index=False)

lstm_forecast_df


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


,ds,lstm_pred
0,2023-01-01,403175.03125
1,2023-02-01,403400.93750
2,2023-03-01,404825.18750
3,2023-04-01,405859.68750
4,2023-05-01,407011.25000
5,2023-06-01,408137.40625


In [23]:
import pandas as pd

# 📥 Load dữ liệu từ file gốc
df = pd.read_parquet("/mnt/data/merged_real_estate_macro.parquet")

# 🕒 Tạo cột ngày chuẩn
df['ds'] = pd.to_datetime(df[['Year', 'Month']].assign(DAY=1))


In [24]:
# 📊 Tính trung bình giá theo tháng
monthly_avg = df.groupby('ds')['Price'].mean().reset_index()
monthly_avg.rename(columns={'Price': 'actual'}, inplace=True)

In [25]:
# 📅 Lọc ra 6 tháng đầu năm 2023
actual_6m = monthly_avg[(monthly_avg['ds'] >= '2023-01-01') & (monthly_avg['ds'] <= '2023-06-01')]

In [26]:
# 💾 Lưu vào file chuẩn
actual_6m.to_csv("/mnt/data/actual_2023.csv", index=False)
print("✅ Đã lưu file actual vào: /mnt/data/actual_2023.csv")

✅ Đã lưu file actual vào: /mnt/data/actual_2023.csv


In [29]:
# ✅ Load tất cả các file dự báo & actual mới
df_prophet = pd.read_csv("/mnt/data/predict_prophet_v3.csv")
df_lstm = pd.read_csv("/mnt/data/predict_lstm_v2.csv")
df_actual = pd.read_csv("/mnt/data/actual_2023.csv")

# ✅ Đảm bảo cột ngày là datetime
df_prophet['ds'] = pd.to_datetime(df_prophet['ds'])
df_lstm['ds'] = pd.to_datetime(df_lstm['ds'])
df_actual['ds'] = pd.to_datetime(df_actual['ds'])

# ✅ Merge 3 bảng theo cột ds
df_merge = df_prophet.merge(df_lstm, on='ds', how='inner')
df_merge = df_merge.merge(df_actual, on='ds', how='inner')
df_merge.rename(columns={"yhat": "prophet_pred"}, inplace=True)

# ✅ Tính hybrid forecast
df_merge['hybrid_mean'] = (df_merge['prophet_pred'] + df_merge['lstm_pred']) / 2
df_merge['hybrid_weighted'] = 0.7 * df_merge['prophet_pred'] + 0.3 * df_merge['lstm_pred']

# ✅ Đánh giá mô hình
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))  # ✅ FIXED
    return {"Model": name, "MAE": mae, "RMSE": rmse}


results = [
    evaluate("Prophet", df_merge["actual"], df_merge["prophet_pred"]),
    evaluate("LSTM", df_merge["actual"], df_merge["lstm_pred"]),
    evaluate("Hybrid Mean", df_merge["actual"], df_merge["hybrid_mean"]),
    evaluate("Hybrid Weighted", df_merge["actual"], df_merge["hybrid_weighted"]),
]

results_df = pd.DataFrame(results)
df_merge.set_index("ds", inplace=True)

results_df, df_merge.tail(6)


(             Model           MAE          RMSE
 0          Prophet  45625.442427  45952.477381
 1             LSTM  46443.134551  46970.865131
 2      Hybrid Mean  46034.288489  46430.259863
 3  Hybrid Weighted  45870.750064  46231.493326,
              prophet_pred  lstm_pred         actual    hybrid_mean  \
 ds                                                                   
 2023-01-01  404848.516175  403175.03  368369.978740  404011.773087   
 2023-02-01  398308.367070  403400.94  350133.900294  400854.653535   
 2023-03-01  408222.643583  404825.20  357314.916104  406523.921791   
 2023-04-01  402610.574678  405859.70  355669.536658  404235.137339   
 
             hybrid_weighted  
 ds                           
 2023-01-01    404346.470322  
 2023-02-01    399836.138949  
 2023-03-01    407203.410508  
 2023-04-01    403585.312274  )

In [30]:
from google.cloud import storage
import os

bucket_name = "boothill2001-dataset"
local_to_gcs = {
    "/mnt/data/predict_prophet_v3.csv": "uk_property_data/forecast/predict_prophet_v3.csv",
    "/mnt/data/predict_lstm_v2.csv": "uk_property_data/forecast/predict_lstm_v2.csv",
    "/mnt/data/actual_2023.csv": "uk_property_data/forecast/actual_2023.csv"
}

client = storage.Client()
bucket = client.bucket(bucket_name)

for local, gcs_path in local_to_gcs.items():
    blob = bucket.blob(gcs_path)
    blob.upload_from_filename(local)
    print(f"✅ Uploaded {local} → gs://{bucket_name}/{gcs_path}")


✅ Uploaded /mnt/data/predict_prophet_v3.csv → gs://boothill2001-dataset/uk_property_data/forecast/predict_prophet_v3.csv
✅ Uploaded /mnt/data/predict_lstm_v2.csv → gs://boothill2001-dataset/uk_property_data/forecast/predict_lstm_v2.csv
✅ Uploaded /mnt/data/actual_2023.csv → gs://boothill2001-dataset/uk_property_data/forecast/actual_2023.csv
